#### Importing Libraries

In [1]:
# Data manipulation and numerical operations
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

# Sparse matrix operations
from scipy.sparse import load_npz

# Model development
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

# Model selection and hyperparameter tuning
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Experiment tracking
import mlflow
import mlflow.sklearn

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

print("All model development libraries imported successfully.")

All model development libraries imported successfully.


#### Loading Feature Engineering Artifacts

In [2]:
# Defining artifact directories
features_dir = Path("../data/processed/features")
models_dir = Path("../models")

# Loading word-level TF-IDF matrices
X_train_word = load_npz(features_dir / "X_train_word.npz")
X_test_word = load_npz(features_dir / "X_test_word.npz")

# Loading character-level TF-IDF matrices
X_train_char = load_npz(features_dir / "X_train_char.npz")
X_test_char = load_npz(features_dir / "X_test_char.npz")

# Loading combined TF-IDF matrices
X_train_combined = load_npz(features_dir / "X_train_combined.npz")
X_test_combined = load_npz(features_dir / "X_test_combined.npz")

# Loading encoded target labels
y_train = np.load(features_dir / "y_train.npy")
y_test = np.load(features_dir / "y_test.npy")

# Loading fitted transformers
word_vectorizer = joblib.load(models_dir / "word_tfidf_vectorizer.joblib")
char_vectorizer = joblib.load(models_dir / "char_tfidf_vectorizer.joblib")
label_encoder = joblib.load(models_dir / "label_encoder.joblib")

# Displaying loaded artifact dimensions
print("Feature Representation       Training Shape       Testing Shape")
print("-" * 68)

for name, X_train, X_test in [
    ("Word TF-IDF", X_train_word, X_test_word),
    ("Character TF-IDF", X_train_char, X_test_char),
    ("Combined TF-IDF", X_train_combined, X_test_combined)
]:
    print(f"{name:<28} {str(X_train.shape):<20} {X_test.shape}")

print(f"\nTraining target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")
print(f"Number of categories: {len(label_encoder.classes_)}")
print("\nAll feature engineering artifacts loaded successfully.")

Feature Representation       Training Shape       Testing Shape
--------------------------------------------------------------------
Word TF-IDF                  (28919, 30000)       (6391, 30000)
Character TF-IDF             (28919, 50000)       (6391, 50000)
Combined TF-IDF              (28919, 80000)       (6391, 80000)

Training target shape: (28919,)
Testing target shape: (6391,)
Number of categories: 10

All feature engineering artifacts loaded successfully.


**Observations on Loaded Feature Engineering Artifacts:**

- All three TF-IDF representations were loaded successfully, containing **30,000 word features**, **50,000 character features**, and **80,000 combined features**.
- The training and testing datasets contain **28,919** and **6,391** records, respectively, across **10 product categories**.
- The saved feature matrices, encoded targets, and fitted transformers are available for reproducible model development without repeating feature extraction.

#### Validating Training Data and Group Integrity

In [3]:
# Loading original training and testing records
train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

# Extracting group identifiers
train_groups = train_df["Split_Group"].to_numpy()
test_groups = test_df["Split_Group"].to_numpy()

# Validating feature and target alignment
for X_train, X_test in [
    (X_train_word, X_test_word),
    (X_train_char, X_test_char),
    (X_train_combined, X_test_combined)
]:
    assert X_train.shape[0] == len(y_train) == len(train_groups)
    assert X_test.shape[0] == len(y_test) == len(test_groups)
    assert X_train.shape[1] == X_test.shape[1]

# Verifying encoded labels match the original records
assert np.array_equal(
    label_encoder.transform(train_df["Category_Label"]),
    y_train
)

assert np.array_equal(
    label_encoder.transform(test_df["Category_Label"]),
    y_test
)

# Verifying that groups do not overlap across partitions
train_group_set = set(train_groups)
test_group_set = set(test_groups)

assert train_group_set.isdisjoint(test_group_set)

# Displaying validation results
print(f"Training records: {len(train_df):,}")
print(f"Testing records: {len(test_df):,}")
print(f"Training groups: {len(train_group_set):,}")
print(f"Testing groups: {len(test_group_set):,}")
print(f"Overlapping groups: {len(train_group_set & test_group_set)}")
print(f"Number of target classes: {len(np.unique(y_train))}")

print("\nFeature alignment and group integrity verified successfully.")

Training records: 28,919
Testing records: 6,391
Training groups: 9,577
Testing groups: 2,395
Overlapping groups: 0
Number of target classes: 10

Feature alignment and group integrity verified successfully.


**Observations on Training Data and Group Integrity:**

- The training and testing partitions contain **28,919** and **6,391** records, respectively, with consistent feature and target alignment.
- The dataset contains **9,577 training groups** and **2,395 testing groups**, with **zero overlapping groups** between partitions.
- All **10 product categories** are represented in the training data, supporting multiclass model development and group-aware cross-validation.

#### Configuring MLflow Experiment Tracking

In [4]:
# Defining the MLflow tracking database
tracking_db = Path("../mlflow.db").resolve()

# Configuring MLflow tracking
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")

# Creating or selecting the project experiment
experiment_name = "Product Taxonomy Intelligence"

mlflow.set_experiment(experiment_name)

# Displaying experiment information
experiment = mlflow.get_experiment_by_name(experiment_name)

print(f"Experiment name: {experiment.name}")
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Artifact location: {experiment.artifact_location}")

print("\nMLflow experiment tracking configured successfully.")

2026/10/28 20:59:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/28 20:59:27 INFO mlflow.store.db.utils: Updating database tables
2026/10/28 20:59:30 INFO mlflow.tracking.fluent: Experiment with name 'Product Taxonomy Intelligence' does not exist. Creating a new experiment.


Experiment name: Product Taxonomy Intelligence
Experiment ID: 1
Tracking URI: sqlite:///D:/Projects/ML and Data Science/Product Taxonomy Intelligence/mlflow.db
Artifact location: file:///d:/Projects/ML and Data Science/Product Taxonomy Intelligence/notebooks/mlruns/1

MLflow experiment tracking configured successfully.


#### Establishing the Majority-Class Baseline

In [5]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

# Configuring group-aware cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Defining baseline classifier
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Defining evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}

# Evaluating baseline using training data only
baseline_cv = cross_validate(
    estimator=baseline_model,
    X=X_train_word,
    y=y_train,
    groups=train_groups,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

# Summarizing cross-validation results
baseline_results = {
    "Model": "Majority Class Baseline",
    "Feature_Representation": "Word TF-IDF",
    "Accuracy": baseline_cv["test_accuracy"].mean(),
    "Macro_F1": baseline_cv["test_f1_macro"].mean(),
    "Weighted_F1": baseline_cv["test_f1_weighted"].mean()
}

baseline_results_df = pd.DataFrame([baseline_results])

display(baseline_results_df.round(4))

print("\nCross-validation completed successfully.")

,Model,Feature_Representation,Accuracy,Macro_F1,Weighted_F1
0,Majority Class Baseline,Word TF-IDF,0.158,0.0273,0.0431



Cross-validation completed successfully.


**Observations on Majority-Class Baseline:**

- The majority-class baseline achieved **15.80% accuracy**, reflecting the limited performance of predicting only the most frequent product category.
- The **Macro F1-score of 0.0273** indicates extremely poor classification performance across the 10 product categories.
- The baseline establishes a minimum benchmark for evaluating whether subsequent machine learning models learn meaningful product classification patterns.

#### Logging Baseline Performance in MLflow

In [6]:
# Logging baseline experiment
with mlflow.start_run(run_name="Baseline_Majority_Class"):

    # Logging model configuration
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("feature_representation", "Word TF-IDF")
    mlflow.log_param("cv_strategy", "StratifiedGroupKFold")
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("random_state", RANDOM_STATE)

    # Logging cross-validation metrics
    mlflow.log_metric("cv_accuracy_mean", baseline_cv["test_accuracy"].mean())
    mlflow.log_metric("cv_macro_f1_mean", baseline_cv["test_f1_macro"].mean())
    mlflow.log_metric("cv_weighted_f1_mean", baseline_cv["test_f1_weighted"].mean())

    mlflow.log_metric("cv_accuracy_std", baseline_cv["test_accuracy"].std())
    mlflow.log_metric("cv_macro_f1_std", baseline_cv["test_f1_macro"].std())
    mlflow.log_metric("cv_weighted_f1_std", baseline_cv["test_f1_weighted"].std())

    # Logging baseline results as an artifact
    mlflow.log_table(
        baseline_results_df,
        artifact_file="baseline_results.json"
    )

print("Majority-class baseline logged successfully in MLflow.")

Majority-class baseline logged successfully in MLflow.
